<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/12_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt"
filename = 'names.txt'

with urllib.request.urlopen(url) as f:
    data = f.read().decode("utf-8")

words= data.splitlines()


In [12]:
chars=sorted(set(".".join(words)))
stoi={j:i for i,j in enumerate(chars)}
print(chars)
stoi["."]

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


0

In [13]:
context=3
words=["."*context+word+"."for word in words]

In [239]:
import random

random.seed(42)
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
X_tr=[word[i:i+context] for word in words[:n1] for i in range(0,len(word)-context)]
X_dev=[word[i:i+context] for word in words[n1:n2] for i in range(0,len(word)-context)]
X_test=[word[i:i+context] for word in words[n2:] for i in range(0,len(word)-context)]

X_tr=[stoi[x] for val in X_tr for x in val]
X_dev=[stoi[x] for val in X_dev for x in val]
X_test=[stoi[x] for val in X_test for x in val]

X_tr=torch.tensor(X_tr).reshape(-1,3)
X_dev=torch.tensor(X_dev).reshape(-1,3)
X_test=torch.tensor(X_test).reshape(-1,3)

Y_tr=[stoi[word[i+context]] for word in words  for i in range(0,len(word)-context)]
Y_dev=[stoi[word[i+context]] for word in words[n1:n2]  for i in range(0,len(word)-context)]
Y_test=[stoi[word[i+context]] for word in words[n2:]  for i in range(0,len(word)-context)]
Y_tr=torch.tensor(Y_tr)
Y_dev=torch.tensor(Y_dev)
Y_test=torch.tensor(Y_test)

In [14]:
#dont think below is necessary looking back

#Make mappings of context of 3 to next predicted letter. For each word. Aka data X and data Y.
#lets replace whole data with numbers
num_words=[stoi[i] for word in words for i in word]
num_words[:5]

[0, 0, 0, 5, 13]

In [118]:
X_str=[word[i:i+context] for word in words for i in range(0,len(word)-context)]
Y=[stoi[word[i+context]] for word in words  for i in range(0,len(word)-context)]

X=[stoi[x] for val in X_str for x in val]
X=torch.tensor(X).reshape(-1,3)
Y=torch.tensor(Y)


In [16]:
#made respectable

In [287]:
g=torch.Generator().manual_seed(2147483647)
C=torch.randn((27,2),generator=g)
W1=torch.randn((6,100),generator=g)*0.1
b1=torch.randn((100),generator=g)
W2=torch.randn((100,27),generator=g)*0.1
b2=torch.randn((27),generator=g)
parameters=[C, W1,b1,W2,b2]

In [288]:
for p in parameters:
  p.requires_grad=True

In [289]:
lrs=torch.linspace(-3,0,1000)
lrs=10**lrs
lossi=[]

In [291]:
for i in range (5000):
  id=torch.randint(0, len(X_tr), (32,), generator=g)
  emb=C[X_tr[id]]
  h=torch.tanh(emb.view(-1,6)@W1 +b1)
  logits=h@W2+b2
  loss = F.cross_entropy(logits,Y_tr[id])

  for p in parameters:
    p.grad=None #Clearing gradients
  loss.backward()

  lr=-0.1
  for p in parameters:
    p.data+= lr*p.grad
#  lossi.append(loss.item())
print(loss.item())

2.3060312271118164


tensor([[-0.9348,  1.0000,  0.9258,  ...,  0.9786, -0.1926,  0.9515],
        [-0.8512, -0.6315, -0.2024,  ...,  0.5126,  0.9976, -0.8357],
        [-0.9783,  1.0000, -0.8888,  ..., -0.9071,  0.8820,  0.8329],
        ...,
        [ 0.8435,  0.9924, -0.9927,  ...,  0.7392,  0.9822,  0.9999],
        [ 0.8884,  0.6995, -0.2275,  ...,  0.9824,  0.9997,  1.0000],
        [-0.9962,  0.9985, -0.9830,  ..., -0.9998, -0.2699, -0.9489]],
       grad_fn=<TanhBackward0>)

In [238]:
X_tr

[0,
 0,
 0,
 0,
 0,
 20,
 0,
 20,
 1,
 20,
 1,
 21,
 1,
 21,
 18,
 21,
 18,
 5,
 18,
 5,
 14,
 0,
 0,
 0,
 0,
 0,
 19,
 0,
 19,
 21,
 19,
 21,
 12,
 21,
 12,
 5,
 12,
 5,
 13,
 5,
 13,
 1,
 13,
 1,
 14,
 0,
 0,
 0,
 0,
 0,
 26,
 0,
 26,
 5,
 26,
 5,
 18,
 5,
 18,
 5,
 18,
 5,
 14,
 5,
 14,
 9,
 14,
 9,
 20,
 9,
 20,
 25,
 0,
 0,
 0,
 0,
 0,
 8,
 0,
 8,
 1,
 8,
 1,
 22,
 1,
 22,
 1,
 22,
 1,
 14,
 1,
 14,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 19,
 1,
 19,
 9,
 19,
 9,
 6,
 0,
 0,
 0,
 0,
 0,
 11,
 0,
 11,
 5,
 11,
 5,
 18,
 5,
 18,
 18,
 18,
 18,
 9,
 18,
 9,
 3,
 9,
 3,
 11,
 0,
 0,
 0,
 0,
 0,
 13,
 0,
 13,
 1,
 13,
 1,
 4,
 1,
 4,
 4,
 4,
 4,
 21,
 4,
 21,
 24,
 0,
 0,
 0,
 0,
 0,
 10,
 0,
 10,
 9,
 10,
 9,
 18,
 9,
 18,
 5,
 18,
 5,
 8,
 0,
 0,
 0,
 0,
 0,
 18,
 0,
 18,
 8,
 18,
 8,
 5,
 8,
 5,
 14,
 5,
 14,
 14,
 0,
 0,
 0,
 0,
 0,
 13,
 0,
 13,
 9,
 13,
 9,
 12,
 9,
 12,
 4,
 12,
 4,
 18,
 4,
 18,
 5,
 18,
 5,
 4,
 0,
 0,
 0,
 0,
 0,
 19,
 0,
 19,
 5,
 19,
 5,
 9,
 5,
 9,
 12,
 9,


In [296]:
#Total loss
emb=C[X_dev]
h=torch.tanh(emb.view(-1,6)@W1 +b1)
logits=h@W2+b2
loss=F.cross_entropy(logits, Y_dev)
loss.item()

2.3776984214782715

tensor([[ 1.5224e+00, -1.6916e+00, -1.7207e+00, -1.5940e+00, -9.6797e-01,
         -1.3441e+00, -2.2992e+00, -2.8230e+00,  3.1602e-01, -1.0577e-01,
         -2.7357e+00, -1.8040e+00,  2.1864e-01, -1.4448e+00,  6.5573e-01,
         -2.2386e+00, -2.5364e+00, -3.3537e+00,  3.4021e-01, -4.1181e-01,
         -1.3145e+00, -4.1876e+00, -9.8820e-01, -2.1102e+00, -3.2356e+00,
         -2.9151e-01, -2.4425e+00],
        [-2.3087e+00, -6.1290e-01,  9.6199e-02, -1.9699e+00,  1.3246e+00,
         -1.4644e+00, -3.4723e+00, -8.3209e-01,  4.3177e-01, -2.8200e-01,
         -2.2436e+00, -7.1419e-01,  1.5761e+00,  3.2406e-01,  7.3330e-01,
         -2.3322e+00, -2.1718e+00, -4.1673e+00,  1.1228e+00,  7.5714e-01,
         -8.8259e-01, -1.9159e+00,  9.3193e-01, -1.2599e+00, -1.3798e+00,
         -5.5793e-01, -7.3091e-01],
        [-1.9606e+00,  3.3371e+00,  2.4129e+00,  1.9297e+00,  2.3579e+00,
          1.6332e+00,  4.1914e-01,  1.4400e+00,  1.7816e+00,  1.6342e+00,
          2.6786e+00,  3.0597e+00,  2.17

In [136]:
torch.randint(0, len(X), (1, 32), generator=g).view(-1)

tensor([ 49774, 201075, 212273, 187393,  20354,  63832,  22730, 107031, 164006,
        205346, 126142, 163533, 133320, 125742,  15934, 146878,  44727, 138516,
        156645, 214811, 225830, 166871, 123348,  91466,  54705, 103478, 151030,
        177184, 206696,  81496,   3489, 119286])

TypeError: 'Tensor' object is not callable